# DeepSeekMoE — Fine-Grained Expert Segregation

源码导航：[core/ffn/moe_deepseek.py](../../../core/ffn/moe_deepseek.py) 中的 `DeepSeekMoE`。

Dai et al. (2024) 在 *DeepSeek-V2* 中提出将 MoE 专家分为两类：
- **Shared experts**（共享专家）：所有 token 始终激活，负责通用高频知识；
- **Routed experts**（路由专家）：通过 top-k 路由按需激活，负责特定低频模式。

这种分离使 routed 专家可以更专注地学习细粒度模式，而 shared 专家承担基础语言建模，显著提升了专家利用率和模型效率。

### 1. 理论推导

#### 1.1 输出公式

$$y = \underbrace{\sum_{s=1}^{S} \text{Shared}_s(x)}_{\text{always active}} + \underbrace{\sum_{r \in \text{topk}} g_r \cdot \text{Routed}_r(x)}_{\text{conditionally active}}$$

其中 $S$ 为共享专家数（通常 2），routed 专家数 $R$ 可达 64–256。

#### 1.2 与基础 MoE 的核心差异

| 维度 | TopKMoE | DeepSeekMoE |
|---|---|---|
| 专家分类 | 统一专家池 | 共享 + 路由分离 |
| 始终激活 | 无 | $S$ 个共享专家 |
| 路由范围 | 全部 $E$ 个专家 | 仅 routed 专家 ($R$) |
| 专家粒度 | 粗（如 8×） | 细（如 2 shared + 64 routed） |
| 负载均衡 | 全局均匀 | 仅 routed 专家均匀 |
| 专家利用率 | 较低 | 更高（shared 兜底通用知识） |

#### 1.3 细粒度专家的理论动机

传统 MoE 的专家是"全能型"的：每个专家需要同时学习通用知识和特定知识。DeepSeekMoE 将通用知识提取到共享专家中，使 routed 专家只需学习**残差补充知识**。这等价于在 MoE 中引入了一种结构化的"基础 + 增量"分解。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.moe_deepseek import DeepSeekMoE

### 2. 形状与接口验证

In [ ]:
torch.manual_seed(0)
moe = DeepSeekMoE(
    n_embd=128,
    num_shared_experts=2,
    num_routed_experts=16,
    top_k=4,
    d_ffn=256,
    capacity_factor=1.0,
    aux_loss_coef=0.01,
)

x = torch.randn(2, 16, 128)
y, aux_loss = moe(x)

print(f"输入形状:   {tuple(x.shape)}")
print(f"输出形状:   {tuple(y.shape)}")
print(f"aux_loss:   {aux_loss.item():.6f}")
print(f"共享专家数: {len(moe.shared_experts)}")
print(f"路由专家数: {len(moe.routed_experts)}")
assert x.shape == y.shape, "DeepSeekMoE 必须保持维度一致！"

### 3. 共享专家始终激活验证

In [ ]:
torch.manual_seed(42)
x = torch.randn(1, 8, 64)
moe = DeepSeekMoE(
    n_embd=64, num_shared_experts=2, num_routed_experts=8,
    top_k=2, d_ffn=128, capacity_factor=None
)

# 单独计算共享专家输出
x_flat = x.view(-1, 64)
shared_out = torch.zeros_like(x_flat)
for expert in moe.shared_experts:
    shared_out += expert(x_flat)

shared_out = shared_out.view(1, 8, 64)

# 完整前向
full_out, _ = moe(x)

print(f"共享专家输出均值: {shared_out.mean().item():.4f}")
print(f"完整输出均值:     {full_out.mean().item():.4f}")
print(f"差异:             {(full_out - shared_out).abs().mean().item():.4f}")
差异即为 routed 专家贡献，证明共享专家始终参与计算。

### 4. 路由分布观察

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
x = torch.randn(1, 64, 64)
moe = DeepSeekMoE(
    n_embd=64, num_shared_experts=2, num_routed_experts=16,
    top_k=4, d_ffn=128, capacity_factor=None
)

# 提取 router 概率
x_flat = x.view(-1, 64)
logits = moe.router(x_flat)
probs = torch.softmax(logits, dim=-1).detach()  # (64, 16)

# 每个 token 的 top-4 专家分布
topk_vals, topk_idx = torch.topk(probs, k=4, dim=-1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, ax in enumerate(axes.flat):
    ax.bar(range(16), probs[idx].numpy(), color="steelblue")
    ax.set_title(f"Token {idx} 的路由概率")
    ax.set_xlabel("Routed Expert ID")
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1)
    for k, (v, e) in enumerate(zip(topk_vals[idx], topk_idx[idx])):
        ax.annotate(f"e{e}", (e, v), ha="center", va="bottom", fontsize=8)

plt.suptitle("DeepSeekMoE Router Distribution (first 4 tokens)", fontsize=14)
plt.tight_layout()
plt.show()

每个 token 的路由概率集中在少数几个专家上，体现稀疏性。

### 5. 参数量对比：TopKMoE vs DeepSeekMoE

In [ ]:
from core.ffn.moe_base import TopKMoE

n_embd, d_ffn = 128, 256

# TopKMoE: 8 experts
moe_base = TopKMoE(n_embd=n_embd, num_experts=8, top_k=2, d_ffn=d_ffn)
base_total = sum(p.numel() for p in moe_base.parameters())

# DeepSeekMoE: 2 shared + 8 routed
moe_ds = DeepSeekMoE(
    n_embd=n_embd, num_shared_experts=2,
    num_routed_experts=8, top_k=2, d_ffn=d_ffn
)
ds_total = sum(p.numel() for p in moe_ds.parameters())

print(f"TopKMoE (8 experts):     {base_total:>10,d} params")
print(f"DeepSeekMoE (2s+8r):     {ds_total:>10,d} params")
print(f"DeepSeekMoE 激活:        {ds_total - sum(p.numel() for p in moe_ds.routed_experts.parameters()):>10,d} (shared 始终激活)")

expert_p = sum(p.numel() for p in moe_base.experts[0].parameters())
base_active = n_embd * 8 + 2 * expert_p
ds_active = sum(p.numel() for p in moe_ds.shared_experts.parameters()) + n_embd * 8 + 2 * expert_p
print(f"\nTopKMoE 激活参数量:     {base_active:>10,d}")
print(f"DeepSeekMoE 激活参数量:   {ds_active:>10,d}")

### 6. 源码精讲

```python
class DeepSeekMoE(nn.Module):
    def __init__(self, n_embd, num_shared=2, num_routed=64, top_k=6, ...):
        super().__init__()
        self.shared_experts = nn.ModuleList([... for _ in range(num_shared)])
        self.routed_experts = nn.ModuleList([... for _ in range(num_routed)])
        self.router = nn.Linear(n_embd, num_routed, bias=False)

    def forward(self, x):
        # 1. Shared experts (always active)
        shared_out = sum(e(x) for e in self.shared_experts)

        # 2. Routed experts (top-k selected)
        router_probs = softmax(self.router(x), dim=-1)
        top_k_weights, top_k_indices = torch.topk(router_probs, k)
        routed_out = dispatch_and_aggregate(x, top_k_weights, top_k_indices)

        # 3. Output = shared + routed
        return shared_out + routed_out, aux_loss
```

关键设计点：
- 共享专家数量极少（2–4），参数量可控，始终激活不影响稀疏性。
- 路由网络仅作用于 routed 专家，共享专家不参与路由竞争。
- 输出为共享专家之和加上路由专家的加权聚合，形成"基础 + 增量"结构。
- 负载均衡损失仅约束 routed 专家的分配，共享专家自然均匀覆盖所有 token。

---

## 延伸阅读与参考资料

### 核心论文
- **DeepSeek-V2**: Dai et al., 2024. [arXiv:2405.04434](https://arxiv.org/abs/2405.04434)
- **DeepSeek-V3**: Liu et al., 2024. [arXiv:2412.19437](https://arxiv.org/abs/2412.19437)

### 工程实践
- DeepSeek-V2 采用 2 shared + 64 routed 专家，总参数量 236B，激活参数量 21B。
- DeepSeek-V3 扩展到 1 shared + 256 routed 专家，总参数量 671B，激活参数量 37B。
- 共享专家机制被 Qwen3 等后续模型借鉴。